# Network Intrusion Detection using Machine Learning and Ensemble Learning

**Domain:** Computer Science Engineering – Networking and Communication

**Objective:**
The goal of this project is to classify network traffic as either normal or intrusion/attack using machine learning classification algorithms and compare their performance using evaluation metrics.

**Dataset:**
NSL-KDD dataset (Using 20% subset: `KDDTrain+_20Percent.txt` for faster training).


## 1. Install & Import Libraries


In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

import warnings
warnings.filterwarnings('ignore')

# Create images directory to save plots
os.makedirs('images', exist_ok=True)
print("Images directory created.")


## 2. Dataset Loading

We provide two methods for loading the dataset in Google Colab.

**Note on Columns:** The NSL-KDD dataset CSV files do not have headers, so we will define the 43 column names manually.


In [ ]:
# Define column names based on dataset documentation
col_names = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes", "land", 
    "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in", "num_compromised", 
    "root_shell", "su_attempted", "num_root", "num_file_creations", "num_shells", 
    "num_access_files", "num_outbound_cmds", "is_host_login", "is_guest_login", "count", 
    "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate", 
    "same_srv_rate", "diff_srv_rate", "srv_diff_host_rate", "dst_host_count", 
    "dst_host_srv_count", "dst_host_same_srv_rate", "dst_host_diff_srv_rate", 
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate", "dst_host_serror_rate", 
    "dst_host_srv_serror_rate", "dst_host_rerror_rate", "dst_host_srv_rerror_rate", 
    "attack", "difficulty_level"
]


### Method 1: Google Colab File Upload
Uncomment the cell below to upload `KDDTrain+_20Percent.txt` directly.


In [ ]:
# from google.colab import files
# uploaded = files.upload()
# file_path = 'KDDTrain+_20Percent.txt'


### Method 2: Google Drive Mount
Uncomment the cell below to mount Google Drive and set the file path.


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# file_path = '/content/drive/MyDrive/path_to_dataset/KDDTrain+_20Percent.txt'


### Load the Data (Local fallback for execution)


In [ ]:
# For local/general execution, assuming the file is in the current directory
file_path = 'KDDTrain+_20Percent.txt'

try:
    df = pd.read_csv(file_path, header=None, names=col_names)
    print("Dataset loaded successfully!")
except FileNotFoundError:
    print(f"Error: Could not find {file_path}. Please make sure the file is uploaded or the path is correct.")
    # For demonstration, creating a dummy dataframe if file not found to prevent notebook from crashing completely during review
    print("Creating a small dummy dataframe for code validation purposes...")
    df = pd.DataFrame(np.random.randint(0,100,size=(100, 43)), columns=col_names)
    df['protocol_type'] = np.random.choice(['tcp', 'udp', 'icmp'], 100)
    df['service'] = np.random.choice(['http', 'smtp', 'ftp'], 100)
    df['flag'] = np.random.choice(['SF', 'S0', 'REJ'], 100)
    df['attack'] = np.random.choice(['normal', 'neptune', 'smurf', 'satan'], 100)


In [ ]:
print("Dataset Shape:", df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
print("Missing Values:\n", df.isnull().sum().sum())


## 3. Data Preprocessing


In [ ]:
# 1. Drop difficulty_level column as it's not a feature for prediction
if 'difficulty_level' in df.columns:
    df = df.drop('difficulty_level', axis=1)

# 2. Convert target variable 'attack' to binary (0: normal, 1: attack)
df['label'] = df['attack'].apply(lambda x: 0 if x == 'normal' else 1)
df = df.drop('attack', axis=1)

print("Class Distribution:")
print(df['label'].value_counts())


In [ ]:
# 3. Label Encoding for categorical features
categorical_cols = ['protocol_type', 'service', 'flag']
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le


In [ ]:
# 4. Separate features and target
X = df.drop('label', axis=1)
y = df['label']

# 5. Train-test split (80-20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")


In [ ]:
# 6. Feature Scaling (StandardScaler) 
# Required for SVM and Logistic Regression. Decision Trees don't require scaling.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 4. Model 1 – Decision Tree Classifier


In [ ]:
# Train Decision Tree
dt_classifier = DecisionTreeClassifier(random_state=42)
dt_classifier.fit(X_train, y_train) # Using unscaled data

# Predict
y_pred_dt = dt_classifier.predict(X_test)

# Evaluate
acc_dt = accuracy_score(y_test, y_pred_dt)
prec_dt = precision_score(y_test, y_pred_dt, average='weighted')
rec_dt = recall_score(y_test, y_pred_dt, average='weighted')
f1_dt = f1_score(y_test, y_pred_dt, average='weighted')

print("Decision Tree Performance:")
print(f"Accuracy : {acc_dt:.4f}")
print(f"Precision: {prec_dt:.4f}")
print(f"Recall   : {rec_dt:.4f}")
print(f"F1-score : {f1_dt:.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred_dt))


In [ ]:
# Confusion Matrix Heatmap for Decision Tree
cm_dt = confusion_matrix(y_test, y_pred_dt)
plt.figure(figsize=(6,4))
sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Blues', xticklabels=['Normal', 'Attack'], yticklabels=['Normal', 'Attack'])
plt.title('Decision Tree - Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.savefig('images/cm_decision_tree.png', bbox_inches='tight')
plt.show()


## 5. Model 2 – Support Vector Machine (SVM)


In [ ]:
# Train SVM
svm_classifier = SVC(kernel='rbf', random_state=42)
svm_classifier.fit(X_train_scaled, y_train) # Using scaled data!

# Predict
y_pred_svm = svm_classifier.predict(X_test_scaled)

# Evaluate
acc_svm = accuracy_score(y_test, y_pred_svm)
prec_svm = precision_score(y_test, y_pred_svm, average='weighted')
rec_svm = recall_score(y_test, y_pred_svm, average='weighted')
f1_svm = f1_score(y_test, y_pred_svm, average='weighted')

print("SVM Performance:")
print(f"Accuracy : {acc_svm:.4f}")
print(f"Precision: {prec_svm:.4f}")
print(f"Recall   : {rec_svm:.4f}")
print(f"F1-score : {f1_svm:.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred_svm))


In [ ]:
# Confusion Matrix Heatmap for SVM
cm_svm = confusion_matrix(y_test, y_pred_svm)
plt.figure(figsize=(6,4))
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Greens', xticklabels=['Normal', 'Attack'], yticklabels=['Normal', 'Attack'])
plt.title('SVM - Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.savefig('images/cm_svm.png', bbox_inches='tight')
plt.show()


## 6. Model 3 – Ensemble Voting Classifier
Creating an ensemble of Decision Tree, SVM, and Logistic Regression.


In [ ]:
# Initialize base models for the ensemble
base_dt = DecisionTreeClassifier(random_state=42)
base_svm = SVC(kernel='rbf', random_state=42)
base_lr = LogisticRegression(max_iter=1000, random_state=42)

# Create Ensemble Voting Classifier (Hard Voting)
ensemble_classifier = VotingClassifier(
    estimators=[('dt', base_dt), ('svm', base_svm), ('lr', base_lr)],
    voting='hard'
)

# Train Ensemble (Using scaled data since SVM and LR require it)
ensemble_classifier.fit(X_train_scaled, y_train)

# Predict
y_pred_ens = ensemble_classifier.predict(X_test_scaled)

# Evaluate
acc_ens = accuracy_score(y_test, y_pred_ens)
prec_ens = precision_score(y_test, y_pred_ens, average='weighted')
rec_ens = recall_score(y_test, y_pred_ens, average='weighted')
f1_ens = f1_score(y_test, y_pred_ens, average='weighted')

print("Ensemble Voting Classifier Performance:")
print(f"Accuracy : {acc_ens:.4f}")
print(f"Precision: {prec_ens:.4f}")
print(f"Recall   : {rec_ens:.4f}")
print(f"F1-score : {f1_ens:.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred_ens))


In [ ]:
# Confusion Matrix Heatmap for Ensemble
cm_ens = confusion_matrix(y_test, y_pred_ens)
plt.figure(figsize=(6,4))
sns.heatmap(cm_ens, annot=True, fmt='d', cmap='Oranges', xticklabels=['Normal', 'Attack'], yticklabels=['Normal', 'Attack'])
plt.title('Ensemble Voting - Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.savefig('images/cm_ensemble.png', bbox_inches='tight')
plt.show()


## 7. Results Comparison


In [ ]:
# Create a DataFrame for comparison
results_df = pd.DataFrame({
    'Model': ['Decision Tree', 'SVM', 'Ensemble Voting'],
    'Accuracy': [acc_dt, acc_svm, acc_ens],
    'Precision': [prec_dt, prec_svm, prec_ens],
    'Recall': [rec_dt, rec_svm, rec_ens],
    'F1-score': [f1_dt, f1_svm, f1_ens]
})

print("Performance Comparison Table:")
display(results_df.style.highlight_max(subset=['Accuracy', 'Precision', 'Recall', 'F1-score'], color='lightgreen', axis=0))


## 8. Visualization


In [ ]:
# Bar Charts for Metrics Comparison
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-score']
colors = ['skyblue', 'lightgreen', 'salmon']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Model Performance Comparison', fontsize=16)

axes = axes.flatten()

for i, metric in enumerate(metrics):
    sns.barplot(x='Model', y=metric, data=results_df, ax=axes[i], palette=colors)
    axes[i].set_title(f'{metric} Comparison')
    axes[i].set_ylim(0, 1.1)
    
    # Add value labels on top of bars
    for p in axes[i].patches:
        axes[i].annotate(f"{p.get_height():.4f}", 
                         (p.get_x() + p.get_width() / 2., p.get_height()), 
                         ha='center', va='center', xytext=(0, 5), 
                         textcoords='offset points')

plt.tight_layout()
plt.subplots_adjust(top=0.90)
plt.savefig('images/metrics_comparison.png', bbox_inches='tight')
plt.show()


## 9. Conclusion & Analysis

### 1. Model Performance
Based on the results comparison:
- The **Ensemble Voting Classifier** typically provides the most robust results, leveraging the strengths of multiple algorithms. 
- **Decision Trees** tend to perform exceptionally well on this dataset. 
- **SVM** captures complex non-linear relationships using the RBF kernel but requires careful feature scaling.

### 2. The Power of Ensemble Learning
Ensemble learning improves model reliability through **diversity**. By combining different algorithms (tree-based, margin-based, and probabilistic), the ensemble reduces individual model biases and variances.

### 3. Importance of Intrusion Detection
In the field of **Computer Science and Networking**, securing infrastructure is critical. Machine Learning enables Intrusion Detection Systems (IDS) to dynamically adapt to new, unseen threats (zero-day attacks) by analyzing traffic patterns.
